# DVC + Git + Google Drive



## ¿Cómo interactúan Git y DVC?

La regla de oro de DVC es:
- **Git** se encarga del código, scripts, archivos de configuración, hiperparámetros (`params.yaml`), pipelines (`dvc.yaml`), métricas y los **archivos puntero `.dvc`**.
- **DVC** se encarga del contenido pesado real: datasets tabulares (`.csv`), imágenes, carpetas masivas y artefactos de modelos (`.pkl`, `.onnx`). DVC los sube a **Google Drive**.

```text
  [ Tu Código / Scripts / .dvc ] --------> Se guardan en -------> Git (GitHub / GitLab)
  [ Datasets / Imágenes / Modelos ] -----> Se guardan en -------> DVC (Google Drive Remote)
```

---

## Estructura del Proyecto (Cookiecutter Data Science)

```text
.
├── .git/                    <-- Repositorio de Git
├── .gitignore               <-- Editado automáticamente por DVC para ignorar datos pesados
├── .dvc/                    <-- Configuración interna de DVC
│   └── config               <-- Define el remote de Google Drive (rastreado por Git)
├── params.yaml              <-- Definición central de hiperparámetros (rastreado por Git)
├── dvc.yaml                <-- Definición del pipeline de ML (rastreado por Git)
├── dvc.lock                <-- Estado exacto de la última ejecución (rastreado por Git)
├── data/
│   ├── processed/           <-- Datos limpios procesados
│   └── raw/                 <-- Datasets pesados en bruto (CSVs, imágenes)
│       ├── transactions.csv.dvc   <-- Puntero pequeño (rastreado por Git)
│       └── sample_images.dvc      <-- Puntero pequeño (rastreado por Git)
├── metrics/                 <-- Métricas en JSON y gráficos (rastreado por Git)
├── models/                  <-- Modelos entrenados (rastreados por DVC)
├── notebooks/
│   └── tutorial_dvc.ipynb   <-- Este cuaderno interactivo
└── src/
    ├── data/
    │   └── make_dataset.py  <-- Script de procesamiento de datos
    └── models/
        └── train_model.py   <-- Script de entrenamiento
```

## Paso 1: Instalación de Dependencias e Inicialización de Git + DVC

In [ ]:
# Instalación de DVC con el soporte oficial para Google Drive y paquetes auxiliares
!pip install "dvc[gdrive]" pandas scikit-learn pyyaml pillow

In [ ]:
# Nos aseguramos de estar en la raíz del proyecto si ejecutamos desde la carpeta notebooks/
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    %cd ..
print(f"Directorio de trabajo actual: {os.getcwd()}")

In [ ]:
# 1. Inicializar Git (si aún no está inicializado)
!git init

# 2. Inicializar DVC (crea la carpeta .dvc/ y vincula DVC con Git)
!dvc init

In [ ]:
# 3. Guardar la inicialización de DVC en Git
!git status
!git add .dvc/ .gitignore
!git commit -m "build: initialize DVC in repository"

## Paso 2: Configuración del Remote de Google Drive en DVC y Git

### Instrucciones para obtener el ID de la carpeta en Google Drive:
1. Abre [Google Drive](https://drive.google.com) en tu navegador.
2. Crea una carpeta dedicada (ejemplo: `DVC_Remote_MiProyecto`).
3. Entra a la carpeta y copia el ID de la URL:
   `https://drive.google.com/drive/folders/`**`1A2b3C4d5E6f7G8h9I0j_kLmNoPqRsTuV`**
4. Pega ese ID en la variable `GDRIVE_ID` a continuación.

In [ ]:
# Configurar el remote en DVC
GDRIVE_ID = "<TU_GDRIVE_FOLDER_ID>"  # <-- REEMPLAZAR CON TU ID REAL DE GOOGLE DRIVE

!dvc remote add -d gdrive_remote gdrive://{GDRIVE_ID}
!dvc remote list

In [ ]:
# IMPORTANTE: Registramos la nueva configuración de DVC (.dvc/config) en Git
!git add .dvc/config
!git commit -m "config: set Google Drive as default DVC remote"

## Paso 3: Versionado de Datos (CSVs e Imágenes) con DVC y Git

Crearemos un dataset tabular (`transactions.csv`) y un lote de imágenes (`sample_images/`).

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

# Crear directorios según Cookiecutter Data Science
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/raw/sample_images", exist_ok=True)

# 1. Generar CSV sintético
df_raw = pd.DataFrame({
    'feature_1': np.random.normal(10, 2, 100),
    'feature_2': np.random.uniform(0, 100, 100),
    'is_fraud': np.random.choice([0, 1], size=100, p=[0.8, 0.2])
})
df_raw.to_csv("data/raw/transactions.csv", index=False)
print("✅ CSV creado: data/raw/transactions.csv")

# 2. Generar lote de imágenes sintéticas
for i in range(5):
    img_array = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
    img = Image.fromarray(img_array)
    img.save(f"data/raw/sample_images/image_{i+1}.png")
print("✅ 5 Imágenes creadas en: data/raw/sample_images/")

In [ ]:
# 1. Decirle a DVC que rastree los archivos pesados
!dvc add data/raw/transactions.csv
!dvc add data/raw/sample_images

# ¿Qué ocurrió automáticamente?
# - DVC creó data/raw/transactions.csv.dvc y data/raw/sample_images.dvc
# - DVC agregó las rutas reales a data/raw/.gitignore para que Git NO los suba.

In [ ]:
# 2. Guardar los archivos puntero (.dvc) y el .gitignore actualizado en Git
!git add data/raw/transactions.csv.dvc data/raw/sample_images.dvc data/raw/.gitignore
!git commit -m "feat(data): track raw CSV and image directory pointers with DVC"

## Paso 4: Sincronización Remota (`dvc push` a Google Drive)

Al ejecutar `dvc push` por primera vez:
1. Se abrirá la autenticación OAuth2 de Google en el navegador.
2. Otorga los permisos a DVC y copia el código si la consola lo solicita.

In [ ]:
# Subir los datos reales pesados a Google Drive
!dvc push

## Paso 5: Creación del Pipeline de ML (`params.yaml`, scripts en `src/` y `dvc.yaml`)

In [ ]:
%%writefile params.yaml
prepare:
  split_ratio: 0.25
  random_state: 42

train:
  n_estimators: 150
  max_depth: 6
  learning_rate: 0.01
  target_col: "is_fraud"

In [ ]:
# Script de procesamiento en src/data/make_dataset.py
os.makedirs("src/data", exist_ok=True)
with open("src/data/make_dataset.py", "w", encoding="utf-8") as f:
    f.write("""import pandas as pd
import os

def process_data(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df = pd.read_csv(input_path)
    df = df.dropna()
    df.to_csv(output_path, index=False)
    print(f"✅ Datos procesados en: {output_path}")

if __name__ == '__main__':
    process_data('data/raw/transactions.csv', 'data/processed/clean_transactions.csv')
""")

In [ ]:
# Script de entrenamiento y evaluación en src/models/train_model.py
os.makedirs("src/models", exist_ok=True)
with open("src/models/train_model.py", "w", encoding="utf-8") as f:
    f.write("""import pandas as pd
import yaml
import json
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

with open('params.yaml', 'r') as f:
    params = yaml.safe_load(f)

df = pd.read_csv('data/processed/clean_transactions.csv')
target_col = params['train']['target_col']

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=params['prepare']['split_ratio'],
    random_state=params['prepare']['random_state']
)

clf = RandomForestClassifier(
    n_estimators=params['train']['n_estimators'],
    max_depth=params['train']['max_depth'],
    random_state=params['prepare']['random_state']
)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
metrics = {
    'accuracy': float(accuracy_score(y_test, preds)),
    'precision': float(precision_score(y_test, preds, zero_division=0)),
    'recall': float(recall_score(y_test, preds, zero_division=0)),
    'f1_score': float(f1_score(y_test, preds, zero_division=0))
}

os.makedirs('metrics', exist_ok=True)
with open('metrics/eval.json', 'w') as f:
    json.dump(metrics, f, indent=4)

plots_df = pd.DataFrame({'actual': y_test, 'predicted': preds})
plots_df.to_csv('metrics/plots.csv', index=False)
print("✅ Entrenamiento completado. Métricas en metrics/eval.json")
""")

In [ ]:
%%writefile dvc.yaml
stages:
  preprocess:
    cmd: python src/data/make_dataset.py
    deps:
      - src/data/make_dataset.py
      - data/raw/transactions.csv
    outs:
      - data/processed/clean_transactions.csv

  train:
    cmd: python src/models/train_model.py
    deps:
      - src/models/train_model.py
      - data/processed/clean_transactions.csv
    params:
      - prepare.split_ratio
      - prepare.random_state
      - train.n_estimators
      - train.max_depth
      - train.target_col
    metrics:
      - metrics/eval.json:
          cache: false
    plots:
      - metrics/plots.csv:
          template: confusion
          x: predicted
          y: actual
          cache: false

## Paso 6: Ejecución del Pipeline y Control de Versiones Doble (Git + DVC)

Ejecutamos el pipeline con `dvc repro` y guardamos los estados en Git y DVC.

In [ ]:
# Reproducir el pipeline completo con DVC
!dvc repro

In [ ]:
# Ver las métricas calculadas
!dvc metrics show

In [ ]:
# REGISTRO SINCROIZADO:
# 1. Guardar en Git el pipeline (dvc.yaml), el candado (dvc.lock), los params y las métricas
!git add dvc.yaml dvc.lock params.yaml src/ metrics/ data/processed/.gitignore
!git commit -m "feat(pipeline): complete ML pipeline execution"

# 2. Subir los outputs procesados pesados a Google Drive
!dvc push

### Registro de un Segundo Experimento y Comparación

In [ ]:
%%writefile params.yaml
prepare:
  split_ratio: 0.35
  random_state: 999

train:
  n_estimators: 300
  max_depth: 10
  learning_rate: 0.05
  target_col: "is_fraud"

In [ ]:
# DVC re-ejecutará solo las etapas afectadas por la modificación de params.yaml
!dvc repro

In [ ]:
# Comparar los cambios de parámetros con el último commit de Git
!dvc params diff

In [ ]:
# Comparar los cambios de métricas con el último commit de Git (HEAD)
!dvc metrics diff HEAD

In [ ]:
# Registrar el segundo experimento en Git y DVC
!git add dvc.lock params.yaml metrics/
!git commit -m "experiment: increase n_estimators and max_depth"
!dvc push

## Paso 7: Flujo de Trabajo Colaborativo (Clonar y Restaurar Experimentos)

### 1. Para los integrantes del equipo:
```bash
# Clonar repositorio de Git (sólo trae código y punteros .dvc)
git clone <URL_REPOSITORIO>
cd mi_proyecto

# Descargar los datasets y modelos pesados desde Google Drive
dvc pull
```

### 2. Para volver a un experimento/commit antiguo:
```bash
# Volver al commit anterior en Git
git checkout HEAD~1

# Sincronizar el estado de los datos con ese commit específico
dvc checkout
```